# Job Dataset Cleaning

This notebook cleans the raw `indian-job-market-dataset-2025.xlsx` job listings
dataset: parsing salary text, extracting skills from free-text descriptions,
normalizing locations, and cleaning job titles & company names.

This notebook covers **cleaning only** — no modeling.


## 1. Setup

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from bs4 import BeautifulSoup
import re
import json

In [9]:
!pip install rapidfuzz -q
!pip install pyahocorasick -q
from rapidfuzz import process, fuzz
import ahocorasick

## 2. Load Dataset

In [10]:
job = pd.read_excel('/content/drive/MyDrive/Colab/indian-job-market-dataset-2025.xlsx')

## 3. Initial Exploration

Quick look at the raw data before any cleaning — shape, nulls, dtypes, duplicates.

In [11]:
job.sample(5)

,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
77567,NEED Freshers For DATA ENTRY / Computer Operator,260925026736,INR,7 Days Ago,Khyati Arora,"Computer Knowledge,Computer Operating,Backend ...",0-2 Yrs,1.5-2.5 Lacs PA,"Mumbai, Mumbai Suburban, Mumbai (All Areas)",124945795,NaN,NaN,NEED FRESHERS FOR DATA ENTRY / COMPUTER OPERAT...,150000.0,250000.0,0.0,2.0
53617,"Medical Coder -Jobs For Biotech, Biotechnology...",111121002256,INR,1 Day Ago,Achievers Spot,"Biotechnology Engineers,Biotechnology,Biotech,...",0 Yrs,3.25-4.5 Lacs PA,"Madurai, Dindigul, Chennai",732481,NaN,NaN,<p> </p><p><strong>Position: Medical Coder</st...,325000.0,450000.0,0.0,0.0
76769,Regional Sales Manager - IVD Industry,220925015400,INR,8 Days Ago,Oscar Medicare,"Ivd,Rapid,Immunology,Sales,Diagnostics,Hematol...",5-10 Yrs,Not disclosed,"Ahmedabad, Rajkot, Vadodara",832397,49.0,3.0,"To lead and manage the regional sales team, dr...",0.0,0.0,5.0,10.0
39291,Hiring Freshers For Inbound voice process - Cu...,11025016550,INR,2 Days Ago,Glympse Human Capital Services,"BPO,Inbound,Voice Support,Customer Care,Inboun...",0-5 Yrs,4-4.25 Lacs PA,Hybrid - Bangalore/ Bengaluru(Electronic City +4),7003261,NaN,NaN,Shifts :Rotational <br><br>Qualification : Gra...,400000.0,425000.0,0.0,5.0
65692,Business Development Executive,260925015028,INR,6 Days Ago,Hemadri Sales Corporation Alwar,"Target Achievement,Lead Generation,Sales And M...",1-2 Yrs,2.5-3 Lacs PA,Gurugram,124870615,NaN,NaN,We are seeking target-driven Business Develop...,250000.0,300000.0,1.0,2.0


In [12]:
job.isnull().sum()

,0
title,0
jobId,0
currency,0
jobUploaded,0
companyName,4
tagsAndSkills,571
experience,2105
salary,0
location,0
companyId,0


In [13]:
job.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97929 entries, 0 to 97928
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              97929 non-null  object 
 1   jobId              97929 non-null  int64  
 2   currency           97929 non-null  object 
 3   jobUploaded        97929 non-null  object 
 4   companyName        97925 non-null  object 
 5   tagsAndSkills      97358 non-null  object 
 6   experience         95824 non-null  object 
 7   salary             97929 non-null  object 
 8   location           97929 non-null  object 
 9   companyId          97929 non-null  int64  
 10  ReviewsCount       62677 non-null  float64
 11  AggregateRating    62677 non-null  float64
 12  jobDescription     97929 non-null  object 
 13  minimumSalary      97358 non-null  float64
 14  maximumSalary      97358 non-null  float64
 15  minimumExperience  97358 non-null  float64
 16  maximumExperience  973

In [14]:
job.duplicated().sum()

np.int64(247)

## 4. Remove Duplicates & Drop Unneeded Columns

Drops exact duplicate rows, ID columns not useful for the recommender
(`jobId`, `companyId`, `jobUploaded`), and rows with no company name.

In [15]:
job.drop_duplicates(inplace=True)
job.drop(columns=['jobId', 'companyId', 'jobUploaded'], inplace=True)
job = job.dropna(subset=['companyName']).copy()

## 5. Salary Cleaning

The raw `salary` column is free text (e.g. `"5 - 8 Lacs"`, `"25000 / month"`,
`"Variable: 15%"`, `"Unpaid"`, `"Not disclosed"`). Explore the different
formats first, then extract structured `minimumSalary` / `maximumSalary` /
`variable_percentage` columns.

### 5a. Explore salary text formats

In [16]:
t=job['salary'].str.split().explode()
t[~(t.str.contains('-'))].unique()

array(['Lacs', 'PA', 'Not', 'disclosed', '50,000', 'Unpaid',
       '5,000/month', '2.75', '(Including', 'Variable:', '10.0%)',
       '25.0%)', '60', 'Cr', '12,000/month', '2.5', '30.0%)', '20.0%)',
       '10,000/month', '7,000/month', '30,000/month', '0', '60.0%)', '75',
       '8', '15,000/month', '12,800/month', '1.25', '5.0%)', '50', '70',
       '90', '3', '6,000/month', '40', '7', '17,288/month', '10', '3.5',
       '1', '35', '25,000/month', '13,000/month', '84,000', '2.0%)',
       '20,000/month', '8,000/month', '80', 'Less', 'than', '5,000',
       '100.0%)', '8,100/month', '6.5', '12', '9.0%)', '2', '1.75',
       '17,500/month', '45', '12.0%)', '1.5', '4', '18.0%)', '15.0%)',
       '12,500/month', '5.5', '40.0%)', '18,000/month', '17.0%)',
       '21,000/month', '96', '50.0%)', '20', '40,000/month', '95.0%)',
       '4.5', '2.25', '60,000', '24', '3.25', '4.25', '22.5', '1.8', '25',
       '23,000/month', '3.0%)', '5', '6', '28,000/month', '22.0%)',
       '1.0%)', '85', 

In [17]:
t1=job[job['salary'].str.contains('Lacs')]['salary'].str.split().explode()
t1[~(t1.str.contains('-'))].unique()

array(['Lacs', 'PA', '2.75', '(Including', 'Variable:', '10.0%)',
       '25.0%)', '60', 'Cr', '2.5', '30.0%)', '20.0%)', '60.0%)', '75',
       '8', '1.25', '5.0%)', '50', '70', '90', '3', '40', '7', '10',
       '3.5', '1', '35', '2.0%)', '80', '100.0%)', '6.5', '12', '9.0%)',
       '2', '1.75', '45', '12.0%)', '1.5', '4', '18.0%)', '15.0%)', '5.5',
       '40.0%)', '17.0%)', '96', '50.0%)', '20', '95.0%)', '4.5', '2.25',
       '24', '3.25', '4.25', '22.5', '1.8', '25', '3.0%)', '5', '6',
       '22.0%)', '1.0%)', '85', '6.0%)', '75.0%)', '3.75', '9', '30'],
      dtype=object)

In [18]:
t1=job[job['salary'].str.contains('/')]
t1.sample(3)

,title,currency,companyName,tagsAndSkills,experience,salary,location,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
51792,Hiring For HR (Recruitment) Interns || Onsite ...,INR,Arcis E Services,NaN,NaN,"15,000/month",Gurugram,247.0,3.8,<p><strong>Internship Tenure : 3-6 months ( De...,NaN,NaN,NaN,NaN
61839,Content Writer - Telugu,INR,Worknext,NaN,NaN,"25,000/month",Bengaluru(Sector 4 HSR Layout ),NaN,NaN,Were looking for a Telugu Content Writer Inter...,NaN,NaN,NaN,NaN
43829,desktop Support Intern,INR,Champion Infometrics,NaN,NaN,"10,000/month",Bengaluru(HSR Layout +1),NaN,NaN,Duration: 3 Months Internship<br><br> About th...,NaN,NaN,NaN,NaN


### 5b. Extract variable percentage and min/max salary for `"X / Y"` format

In [19]:
job["variable_percentage"] = (job["salary"].str.extract(r"Variable:\s*([\d.]+)%", expand=False).astype(float))

job.loc[job['salary'].str.contains('/'),'minimumSalary'] = job.loc[job['salary'].str.contains('/'),'salary'].str.extract(r"^(.*?)\/",expand=False).str.replace(',', '', regex=False).astype(float)
job.loc[job['salary'].str.contains('/'),'maximumSalary'] = job.loc[job['salary'].str.contains('/'),'minimumSalary']

### 5c. Check remaining unparsed rows

In [20]:
t3 = job[job['maximumSalary'].isna()]
t3.sample(2)

,title,currency,companyName,tagsAndSkills,experience,salary,location,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience,variable_percentage
87501,Intern Trainee,INR,SLK Software,NaN,NaN,Unpaid,Bengaluru,1260.0,3.2,SLK Software Services Private Limited is looki...,NaN,NaN,NaN,NaN,NaN
71820,Vendor Research & Coordination - Intern,INR,Kalkine Solutions,NaN,NaN,Unpaid,Noida(Sector 16),9.0,3.7,Internship certificate on successful completio...,NaN,NaN,NaN,NaN,NaN


In [21]:
t4 = job[job['salary'].str.contains('Unpaid')]
t4.sample(2)

,title,currency,companyName,tagsAndSkills,experience,salary,location,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience,variable_percentage
1086,Internship,INR,Arya Niwas,NaN,NaN,Unpaid,Jaipur,NaN,NaN,Arya Niwas is looking for Internship to join o...,NaN,NaN,NaN,NaN,NaN
5892,Team Neverest Intern,INR,Alphagrep Securities,NaN,NaN,Unpaid,Mumbai,17.0,3.7,Brilliant problem . - . solving abilities . So...,NaN,NaN,NaN,NaN,NaN


### 5d. Handle `"Unpaid"` postings — set salary to 0

In [22]:
unpaid_mask = job['salary'].str.contains('Unpaid', na=False) & job['minimumSalary'].isna()
job.loc[unpaid_mask, 'minimumSalary'] = 0
job.loc[unpaid_mask, 'maximumSalary'] = 0

## 6. Experience Cleaning

Fill missing experience values with 0.

In [23]:
job['minimumExperience'] = job['minimumExperience'].fillna(0)
job['maximumExperience'] = job['maximumExperience'].fillna(0)

## 7. Salary-Disclosed Flag & Column Cleanup

Flags whether a posting actually discloses a salary (`"Not disclosed"` and
`"Unpaid"` both count as not disclosed), then drops the now-redundant raw
`experience` / `salary` text columns.

In [24]:
job['salary_disclosed'] = ~job['salary'].isin(['Not disclosed', 'Unpaid'])

In [25]:
job.drop(columns=['experience','salary'],inplace=True)

## 8. Cleaning Job Description & Skills

### 8a. Strip HTML from job descriptions, normalize casing

In [26]:
job['jobDescription'] = job['jobDescription'].apply(lambda x: BeautifulSoup(x, 'html.parser').get_text(" ", strip=True) if pd.notna(x) else x)

/tmp/ipykernel_712/1933376704.py:1: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  job['jobDescription'] = job['jobDescription'].apply(lambda x: BeautifulSoup(x, 'html.parser').get_text(" ", strip=True) if pd.notna(x) e

In [27]:
job['tagsAndSkills'] = job['tagsAndSkills'].str.lower().str.strip()
job['jobDescription'] = job['jobDescription'].str.lower().str.strip()


### 8b. Build a skill vocabulary from `tagsAndSkills`

Splits the comma-separated tags column into individual skills and keeps
ones that appear at least 5 times across the dataset — rare/one-off tags
are more likely typos or noise than real recurring skills.

In [28]:
skills = job['tagsAndSkills'].dropna().astype(str).str.split(',').explode().str.strip()

skill_count = skills.value_counts()

skill_vocab_5 = sorted(skill_count[skill_count >= 5].index)

print("Unique skills:", len(skill_count))
print("Skills appearing >= 5 times:", len(skill_vocab_5))

Unique skills: 44077
Skills appearing >= 5 times: 11228


### 8c. Filter that vocabulary down to real skills

Three pieces, built in order:
1. **`description_stop_skills`** — generic/recruitment words that show up as
   "tags" but aren't actual skills (grammar words, job-posting boilerplate,
   generic business nouns, etc.).
2. **`short_skill_whitelist`** — an exception list for short tokens that
   *are* real skills despite being short (`c`, `r`, `sql`, `aws`, ...) —
   without this they'd get caught by a length filter.
3. The build loop — walks the tag vocabulary and keeps a term if it's in
   the whitelist, or if it's a multi-word skill, or if it's a single word
   at least 3 characters long and not a stop word.

In [29]:
description_stop_skills = {
    # Grammar / fragments
    'a','an','the','as','be','by','of','to','in','on','at','or',
    'and','for','with','from','into','over','under','both',

    # Short ambiguous terms
    'm','s','e','t','v','ii','iii','iv','sr','jr','pm','bd','hi',
    'co','do','go','us','it','ca','ab','ug',

    # Recruitment / job language
    'job','job description','job requirements','requirements',
    'responsibilities','responsibility','role','position',
    'candidate','candidates','experience','qualification',
    'qualifications','preferred','profile','vacancy','vacancies',
    'opening','openings','application','apply','interview',

    # Education / employment
    'graduate','graduates','graduation','fresher','freshers',
    'intern','internship','trainee','trainees','diploma','degree',
    'school','college','education','career','placement','program',

    # Contact / platforms / logistics
    'hr','contact','call','email','naukri','linkedin','source',
    'public','location','shift','shifts','time','hours','cab',
    'wifi','laptop','office',

    # Generic organizational words
    'business','company','client','clients','team','work','working',
    'area','field','industry','department','organization',
    'organisation','activity','activities','task','tasks',
    'function','functions','project','projects',

    # Generic actions
    'develop','developed','developing','development',
    'manage','managed','managing','management',
    'handling','handle','lead','leading','ensure','ensuring',
    'provide','providing','maintain','maintaining',
    'perform','performing','make','making','use','used','using',
    'identify','build','building','set','join','support','supporting',

    # Generic capability words
    'knowledge','understanding','ability','skill','skills',
    'proficiency','proficient','familiarity','exposure',
    'expertise','capability','learning','training',

    # Generic descriptive words
    'good','strong','basic','key','important','major','minor',
    'specific','relevant','required','mandatory','necessary',
    'focus','final','top','large','dynamic','complex','current',
    'professional','global','creative','analytical',

    # Generic nouns
    'system','systems','equipment','plant','tool','tools',
    'material','materials','document','documents','documentation',
    'report','reports','information','details','detail',
    'process','processes','environment','performance',
    'quality','service','services','operations','operation',
    'technology','technical','engineering',

    # Generic business terms
    'customer','customers','sales','marketing','finance',
    'accounting','strategy','growth','research','content',
    'design','data','analytics','development','production',

    # Miscellaneous
    'com','etc','front','line','end','start','post','part',
    'benefits','benefit','flexible','culture','travel','offline',
    'open','top','bonus','bus','ride','fashion','general',
    'awareness','subject','saving','devices'
}


In [30]:
short_skill_whitelist = {
    'c','c#','c++','r','ai','ml','sql','java','aws','sap',
    'sre','erp','crm','seo','sem','sas','php','go','ruby',
    'perl','html','css','xml','api','ui','ux','iot','cad',
    'cam','git','linux','unix','azure','excel','mysql',
    'oracle','python','scala','kafka','spark','hadoop',
    'tableau','docker','kubernetes','tensorflow'
}


In [31]:
description_skill_vocab = set()

for skill in skill_vocab_5:

    skill = str(skill).strip().lower()

    if not skill:
        continue

    # Keep known short technical skills
    if skill in short_skill_whitelist:
        description_skill_vocab.add(skill)
        continue

    # Remove known generic words
    if skill in description_stop_skills:
        continue

    # Keep multi-word skills
    if ' ' in skill:
        description_skill_vocab.add(skill)
        continue

    # Keep specialized single-word skills
    if len(skill) >= 3:
        description_skill_vocab.add(skill)


print("Tag vocabulary:", len(skill_vocab_5))
print("Description vocabulary:", len(description_skill_vocab))


Tag vocabulary: 11228
Description vocabulary: 10940


### 8d. Extract skills from free-text job descriptions

Not every posting has structured `tagsAndSkills` — many skills only appear
inside the free-text `jobDescription`. This uses an **Aho-Corasick
automaton** (via `pyahocorasick`) to efficiently search for every known
skill from the vocabulary above, in a single pass over each description,
rather than looping a regex search per skill.

Three parts:
1. Build the automaton from `description_skill_vocab`.
2. `valid_match_boundary()` — filters out false-positive matches: mid-word
   matches, short tokens that are actually roman numerals/section markers
   ("Phase II"), and short tokens that are actually abbreviations ("B.E.").
3. `extract_description_skills()` — runs the automaton over each
   description, keeps only the longest match when matches overlap (prefers
   "machine learning" over just "machine"), and applies it to every row.

In [32]:
# Build Aho-Corasick automaton for job descriptions

A_desc = ahocorasick.Automaton()

for skill in description_skill_vocab:
    A_desc.add_word(skill, skill)

A_desc.make_automaton()


In [33]:
# valid_match_boundary

def valid_match_boundary(text, start, end, skill):

    before = text[start - 1] if start > 0 else ' '
    after = text[end + 1] if end + 1 < len(text) else ' '

    # Normal word boundary

    if before.isalnum() or before == '_':
        return False

    if after.isalnum() or after == '_':
        return False


    if len(skill) <= 2:

        # Text immediately before the match
        left_text = text[:start].rstrip()

        # Last few words before the match
        previous_words = left_text.split()

        previous_word = (
            previous_words[-1]
            if previous_words
            else ''
        )

        # Roman numeral / section markers

        if previous_word in {
            'phase',
            'section',
            'level',
            'grade',
            'class',
            'floor',
            'stage',
            'chapter',
            'part'
        }:
            return False

        # Abbreviation detection
        # Examples:
        # b.e.
        # b.sc.
        # m.sc.
        # m.e.


        surrounding = text[
            max(0, start - 3):
            min(len(text), end + 4)
        ]

        if '.' in surrounding:
            return False

    return True


In [34]:
# Extract skills from job description

def extract_description_skills(text):
    """Find skill mentions in free text, keeping only the longest match
    when matches overlap (e.g. prefer 'machine learning' over 'machine')."""
    if pd.isna(text):
        return []
    text = str(text).lower().strip()
    if not text:
        return []

    matches = []
    for end_idx, skill in A_desc.iter(text):
        start_idx = end_idx - len(skill) + 1
        if valid_match_boundary(text, start_idx, end_idx, skill):
            matches.append((start_idx, end_idx, skill))

    matches.sort(key=lambda m: -(m[1] - m[0] + 1))  # longest first

    selected = []
    for start, end, skill in matches:
        if any(s <= start and end <= e for s, e, _ in selected):
            continue  # already covered by a longer match
        selected.append((start, end, skill))

    selected.sort(key=lambda m: m[0])  # restore original text order
    return list(dict.fromkeys(skill for _, _, skill in selected))

job['descriptionSkills'] = job['jobDescription'].apply(extract_description_skills)


## 9. Location Cleaning

### 9a. Initial location parsing

Lowercases, strips parenthetical area names (keeping just the main city),
splits multi-location postings on `,`/`/`, and removes trailing `+2`-style
suffixes and `"hybrid-"` prefixes.

In [35]:
locations = (job['location'].dropna().astype(str).str.lower().str.strip())  # lower casing

locations_base = (locations.str.replace(r'\([^)]*\)', '', regex=True)) # extracting only main city names and removing area names

locations_split = (locations_base.str.split(r'[,/]', regex=True).explode().str.strip()) # splitting wrt to , /

locations_split = (locations_split.str.replace(r'\s*\+\d+\s*$', '', regex=True).str.strip()) # removing +1,+2,...

locations_split = locations_split[locations_split.notna() & (locations_split != '')] # removing empty strings

locations_split = (locations_split.str.replace(r'^hybrid\s*-\s*', '', regex=True).str.strip()) # removing "hybrid-" since i am only trying to extract the city name


### 9b. Find likely duplicate location spellings

Uses **rapidfuzz** to find pairs of location strings that are highly
similar (≥85 similarity ratio) but not identical — candidates for spelling
inconsistencies to review and fix manually below.

In [36]:
location_counts = (locations_split.value_counts())

unique_locations = location_counts.index.tolist()

suggestions = []
for loc in unique_locations:
    for match, score, _ in process.extract(loc, unique_locations, scorer=fuzz.ratio, limit=5):
        if match != loc and score >= 85:
            suggestions.append((loc, match, score, location_counts[loc], location_counts[match]))

location_suggestions = pd.DataFrame(
    suggestions,
    columns=['location', 'possible_match', 'similarity', 'location_count', 'match_count']
).sort_values(['location', 'similarity'], ascending=[True, False])

### 9c. Manually curated spelling-correction dictionary

Built from reviewing the fuzzy-match suggestions above.

In [37]:
spelling_mistakes = {
    "ahmadnagar": "ahmednagar",
    "alahabad": "allahabad",
    "amaravathi": "amravati",
    "andhara pradesh": "andhra pradesh",
    "andhrapradesh": "andhra pradesh",
    "attivram": "attivaram",
    "banswada": "banswara",
    "barddhaman": "bardhaman",
    "basavangudi": "basavanagudi",
    "bhatinda": "bathinda",
    "begaluru": "bengaluru",
    "bhubaneshwar": "bhubaneswar",
    "chengalpaatu": "chengalpattu",
    "chhatisgarh": "chhattisgarh",
    "chitoor": "chittoor",
    "deogarh": "deoghar",
    "dharmsala": "dharamshala",
    "dharmasala": "dharamshala",
    "elluru": "eluru",
    "ferozpur": "firozpur",
    "gandhagar": "gandhinagar",
    "gurugaon": "gurgaon",
    "hajipir": "hajipur",
    "himatnagar": "himmatnagar",
    "hydreabad": "hyderabad",
    "kalburagi": "kalaburagi",
    "karanataka": "karnataka",
    "karnatak": "karnataka",
    "karnata": "karnataka",
    "karaikkudi": "karaikudi",
    "kanniyakumari": "kanyakumari",
    "maharastra": "maharashtra",
    "mahararashtra": "maharashtra",
    "maldah": "malda",
    "manendragarh": "mahendragarh",
    "mudhole": "mudhol",
    "nagerkoil": "nagercoil",
    "nasik": "nashik",
    "narasapur": "narsapur",
    "nelmangala": "nelamangala",
    "penugonda": "penukonda",
    "pujab": "punjab",
    "punja": "punjab",
    "purnea": "purnia",
    "purna": "purnia",
    "rup nagar": "rupnagar",
    "telengana": "telangana",
    "thoothukkudi": "thoothukudi",
    "tiruchirapalli": "tiruchirappalli",
    "tirupur": "tiruppur",
    "trissur": "thrissur",
    "uttrakhand": "uttarakhand",
    "uttrapradesh": "uttarpradesh",
    "vicarabad": "vikarabad",
    "west bangal": "west bengal",
    "sri city": "sricity",
    "sriganganagar": "sri ganganagar",
    "new town": "newtown",
    "kormangala": "koramangala",
    "hrbr layout": "hrbr layout",
    "s.a.s. nagar": "sas nagar",
    "kohlapur": "kolhapur",
    "kolapur": "kolhapur",
}

### 9d. Apply corrections & remove location names mistakenly extracted as skills

Some location names were incorrectly picked up as "skills" by the
Aho-Corasick extraction in step 8d (e.g. a city matching a short skill
token) — this filters them back out now that the location vocabulary is
finalized.

In [38]:
locations_clean = locations_split.replace(spelling_mistakes)
location_vocab = set(locations_clean.dropna().unique())

job['descriptionSkills'] = job['descriptionSkills'].apply(
    lambda skills_list: [s for s in skills_list if s.strip().lower() not in location_vocab]
)

### 9e. Merge tag-based and description-based skills into `finalSkill`

Uses `tagsAndSkills` when present; falls back to the skills extracted from
`jobDescription` when tags are missing.

In [39]:
tags_as_list = job['tagsAndSkills'].apply(
    lambda x: [s.strip() for s in x.split(',')] if pd.notna(x) else []
)

has_tags = job['tagsAndSkills'].notna() & (job['tagsAndSkills'] != '')
job['finalSkill'] = tags_as_list.where(has_tags, job['descriptionSkills'])

job['descriptionSkills'] = job['descriptionSkills'].apply(lambda skills: ', '.join(skills))
job['finalSkill'] = job['finalSkill'].apply(lambda skills: ', '.join(skills))

### 9f. Final location-cleaning function

Consolidates the location-cleaning steps above (parens, split, `+2` suffix,
`hybrid-` prefix, spelling corrections) into one reusable function, and adds
the `location_clean` column.

In [40]:
def clean_location_string(raw):
    if pd.isna(raw):
        return []

    text = str(raw).lower().strip()
    text = re.sub(r'\([^)]*\)', '', text)          # drop area names in parens
    cities = re.split(r'[,/]', text)

    cleaned = []
    for city in cities:
        city = city.strip()
        city = re.sub(r'\s*\+\d+\s*$', '', city).strip()   # drop "+2" etc.
        city = re.sub(r'^hybrid\s*-\s*', '', city).strip()  # drop "hybrid-" prefix
        if not city:
            continue
        city = spelling_mistakes.get(city, city)             # fix spelling
        cleaned.append(city)

    return cleaned


job['location_clean'] = job['location'].apply(clean_location_string)
job['location_clean'] = job['location_clean'].apply(lambda cities: ', '.join(cities))

## 10. Title & Company Name Cleaning

### 10a. Normalize casing

In [41]:
job['title'] = job['title'].str.strip().str.lower()
job['companyName'] = job['companyName'].str.strip().str.lower()

### 10b. Build a location-removal pattern, then the full title-cleaning function

`location_pattern` is built from the location vocabulary established above,
so any known city/location name inside a job title can be stripped out.

`clean_job_titles()` then handles everything else: recruiter noise phrases
("urgent hiring", "walk-in drive"), salary/CTC mentions (removed as a
**bounded clause**, never "to end of string" — otherwise a title with
salary info at the *start* would get wiped out entirely), experience-year
ranges, trailing dates, separator spacing, and common abbreviations. A
safety-net fallback guarantees a row is never left fully empty just because
a regex over-matched.

In [42]:
location_pattern = re.compile(
    r'(?<!\w)(?:' +
    '|'.join(
        re.escape(location)
        for location in sorted(location_vocab, key=len, reverse=True)
    ) +
    r')(?!\w)'
)

In [43]:

def clean_job_titles(series):
    orig = series.astype(str)
    s = orig.str.lower().str.strip()

    # Remove known location names first, while original spacing still makes
    # word boundaries reliable (before heavy punctuation stripping below)
    if location_pattern is not None:
        s = s.str.replace(location_pattern, ' ', regex=True)

    # Single character-keep pass — covers everything not in noise_phrases below:
    # drops @, {}, _, and any other symbol not explicitly whitelisted
    s = s.str.replace(r"[^a-z0-9\s\-/&,.|]", ' ', regex=True)

    noise_phrases = [
        r'\burgent(ly)?\s*(job\s*)?(opening|requirement|hiring)s?\s*(for)?\b',
        r'\bhiring\s*s?\s*(for)?\b',
        r'\brequired\b',
        r'\bwalk[\s-]?in\b',
        r'\bvacanc(y|ies)\b',
        r'^\s*(drive|interview)\s*[-:]?\s*',
        r'^\s*(opening|job|jobs)\s+for\s+',
        r'\bfor\s+freshers?\b',
        r'\bfreshers?\b',
    ]
    for pattern in noise_phrases:
        s = s.str.replace(pattern, '', regex=True)

    # bounded salary/CTC removal — only the salary clause itself, never to end-of-string
    s = s.str.replace(r'\d+(\.\d+)?\s*(lpa|lakhs?|k)\b\s*(ctc|salary|package)?', '', regex=True)
    s = s.str.replace(r'\bctc\b\s*(upto|up to)?\s*', '', regex=True)
    s = s.str.replace(r'\bsalary\b\s*(upto|up to)?\s*(\d+\w*)?\s*(in hand)?', '', regex=True)
    s = s.str.replace(r'\bin\s+hand\b', '', regex=True)

    s = s.str.replace(r'\(?\d+\s*-\s*\d+\s*years?\s*(experience)?[^)]*\)?', '', regex=True)
    s = s.str.replace(r'\bat\s+on\s+\d{1,2}(st|nd|rd|th)?\s+\w+\b', '', regex=True)
    s = s.str.replace(r'\.{2,}', '.', regex=True)
    s = s.str.replace(r'[|]', ',', regex=True)
    s = s.str.replace(r'\s*([\-/,&])\s*', r' \1 ', regex=True)

    replacements = {
        r'\bsr\.?\b': 'senior', r'\bjr\.?\b': 'junior', r'\basst\.?\b': 'assistant',
        r'\bmgr\.?\b': 'manager', r'\bmgmt\.?\b': 'management', r'\bdept\.?\b': 'department',
        r'\bexec\.?\b': 'executive', r'\bassoc\.?\b': 'associate',
    }
    for pattern, replacement in replacements.items():
        s = s.str.replace(pattern, replacement, regex=True)

    s = s.str.replace(r'\.', ' ', regex=True)
    s = s.str.replace(r'\b(for|at|on|in)\s*$', '', regex=True)
    s = s.str.replace(r'^\s*(for|at|on|in)\b\s*', '', regex=True)
    s = s.str.replace(r'^[\s\-/,&]+|[\s\-/,&]+$', '', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()

    # safety net: never fully wipe a non-empty row
    fallback = orig.str.lower().str.replace(r"[^a-z0-9\s]", ' ', regex=True)
    fallback = fallback.str.replace(r'\s+', ' ', regex=True).str.strip()
    empty_mask = (s == '') & (fallback != '')
    s = s.where(~empty_mask, fallback)

    return s

### 10c. Strip legal company suffixes

Removes `Pvt Ltd`, `LLP`, `Inc`, `Corp`, etc. so the same company isn't
treated as multiple distinct entities.

In [44]:
def strip_legal_suffix(name):
    if pd.isna(name):
        return name
    text = str(name).strip()

    pattern = re.compile(
        r'[\s,\-]*\(?\s*'
        r'('
        r'(private|pvt|prvate|p)\s*\.?\s*(limited|ltd)\.?'   # private limited / pvt ltd
        r'|limited|ltd\.?'                                    # limited / ltd alone
        r'|llp'
        r'|inc\.?'
        r'|llc'
        r'|corp(oration)?\.?'
        r'|co\.?'
        r'|plc'
        r')'
        r'\s*\)?\s*$',
        re.IGNORECASE
    )

    cleaned = pattern.sub('', text).strip()
    cleaned = re.sub(r'[\s,\-]+$', '', cleaned)   # drop trailing separators left behind
    cleaned = re.sub(r'\s{2,}', ' ', cleaned)      # collapse leftover double spaces
    return cleaned if cleaned else text            # never return an empty string

### 10d. Apply title & company cleaning, convert USD to INR, fill missing skills

In [45]:
job['title'] = clean_job_titles(job['title'])

job = job[job['title'].str.len() > 2]

job['companyName'] = job['companyName'].apply(strip_legal_suffix)

job.loc[job['currency']=='USD','minimumSalary'] = job.loc[job['currency']=='USD','minimumSalary']*95.66

job.loc[:,'finalSkill'] = job.loc[:,'finalSkill'].fillna('')

## 11. Cleaned Dataset


In [47]:
job.sample(10)

,title,currency,companyName,tagsAndSkills,location,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience,variable_percentage,salary_disclosed,descriptionSkills,finalSkill,location_clean
58161,azure terraform professional,INR,ideslabs,"load balancing,scale,net development,sql serve...",Hyderabad,NaN,NaN,experience with .net development and deploymen...,0.0,0.0,6.0,9.0,NaN,False,".net, net development, development and deploym...","load balancing, scale, net development, sql se...",hyderabad
64751,italian language expert - lucrative incentives,INR,travomint,"italian,travel sales,italiansales,italian lang...",Noida(Sector 58),296.0,3.5,1.converting the inbound calls into a sales.\...,0.0,0.0,1.0,2.0,NaN,False,"inbound calls, gds, amadeus, airport, flight, ...","italian, travel sales, italiansales, italian l...",noida
86781,trust & safety analyst,INR,accenture,"python,sql,debugging,troubleshooting,json,risk...",Hyderabad,66726.0,3.7,qualifications: bca / any graduation . years o...,0.0,0.0,3.0,5.0,NaN,False,"bca, brand, distribution, policies, chrome, store","python, sql, debugging, troubleshooting, json,...",hyderabad
26871,software development engineer,INR,accenture,"pyspark,data analytics,data processing,airflow...",Bengaluru,66726.0,3.7,minimum . 5 year(s) of experience is required....,0.0,0.0,1.0,4.0,NaN,False,"software development, engineer, test, components","pyspark, data analytics, data processing, airf...",bengaluru
53594,consultant pediatrician job in nabh hospital i...,INR,quadaple mohali,"pediatrics,paediatrics,doctors jobs,pediatrici...",Dharmasala,NaN,NaN,consultant pediatrician job in nabh hospital i...,3000000.0,4000000.0,0.0,5.0,NaN,True,"consultant pediatrician, nabh, hospital, salar...","pediatrics, paediatrics, doctors jobs, pediatr...",dharamshala
1455,sap rar professional,INR,maneva consulting,"sap rar,copa,s/4 hana,brim process,ifrs 16,ord...","Hyderabad, Pune, Bengaluru",NaN,NaN,candidates with bachelors in finance/ cpa / cp...,0.0,0.0,5.0,10.0,NaN,False,"cpa, hana, sap rar, revenue accounting, report...","sap rar, copa, s/4 hana, brim process, ifrs 16...","hyderabad, pune, bengaluru"
85673,sustainability modeling specialist,INR,msci services,"human capital,analytical,corporate,regulatory ...",Mumbai,336.0,3.9,msci esg research is seeking a senior associat...,0.0,0.0,3.0,6.0,NaN,False,"senior, associate, investment, sustainability,...","human capital, analytical, corporate, regulato...",mumbai
68503,executive - finance,INR,gionik human capital solutions,"r2r process,o2c cash application,finance,accou...",Kolkata,NaN,NaN,preferred candidate profile . well versed with...,0.0,0.0,5.0,7.0,NaN,False,"revenue, expenses, p2p, o2c, r2r","r2r process, o2c cash application, finance, ac...",kolkata
61889,senior manager - hrbp,INR,golden opportunities,"hrbp,attrition management,hr business partner...",Hyderabad,NaN,NaN,preferred candidate profile 4 -7 years of expe...,0.0,0.0,4.0,7.0,NaN,False,"hrbp, employee engagement, attrition managemen...","hrbp, attrition management, hr business partn...",hyderabad
5462,social media community manager,INR,itransparity,"social media,media,management",Mumbai,7.0,4.0,itransparity is looking for social media commu...,0.0,0.0,1.0,5.0,NaN,False,"social media, manager, delivery, digital conte...","social media, media, management",mumbai


## 12. Save Cleaned Dataset

In [46]:
#job.to_csv('job_cleaned.csv', index=False)